In [12]:
import os
from langchain_core.globals import set_debug
set_debug(False)
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# For better security, load environment variables from a .env file
# from dotenv import load_dotenv
# load_dotenv()
# Make sure your OPENAI_API_KEY is set in the .env file

# Initialize the Language Model (using ChatOpenAI is recommended)
llm = ChatOpenAI(temperature=0)

# --- Prompt 1: Extract Information ---
prompt_extract = ChatPromptTemplate.from_template(
    "Extract the technical specifications from the following text:\n\n{text_input}"
)

# --- Prompt 2: Transform to JSON (Updated for Grouping) ---
prompt_transform = ChatPromptTemplate.from_template(
    "Transform the following specifications into a JSON object with a single key 'specifications' "
    "containing the keys 'cpu', 'memory', and 'storage'.\n\n{specifications}"
)

# --- Prompt 3: Logistics ---
prompt_logistic = ChatPromptTemplate.from_template(
    "You are a logistics engine. Take the following product specifications JSON: "
    "{specs_json}\n\n"
    "Append a key 'shipping_options' containing a list of 3 providers options (e.g., DHL, FedEx, UPS)"
    "with simulated shipping costs , simulated estimated 'delivery_days'"
    "Additional Append a key 'selected_provider'" 
    "containing 'provider' taken from the input JSON, and simulated 'send_date'" 
    "and 'estimated_date_delivery_received' "
    "in item'estimated_date_delivery_received'consider the shipping options 'delivery_days'"
    "Return the final valid JSON only."
)

# --- Prompt 4: Order Details (Using Passthrough Variables) ---
prompt_order = ChatPromptTemplate.from_template(
    "You are an order processing engine. Take the following product/logistics JSON: "
    "{logistic_json}\n\n"
    "And extract the following information from the current order context:\n"
    "Task: Append a key 'order_details' to the JSON containing these 3 fields. "
    "Order_id , Order_date , Order_status"
    "Return the final valid JSON only."
)

#    "- Order ID: {order_id}\n"
#    "- Date: {order_date}\n"
#    "- Status: {order_status}\n\n"


# --- Build the Chain using LCEL and RunnablePassthrough ---
# RunnablePassthrough.assign() allows us to calculate a new variable
# and add it to the current dictionary 'backpack' without losing previous keys.

full_chain = (
    # Step 1: Extract specs. Input: {text_input, order_*} -> Output adds {specifications}
    RunnablePassthrough.assign(specifications=prompt_extract | llm | StrOutputParser())
    
    # Step 2: Transform to JSON. Input includes {specifications} -> Output adds {specs_json}
    | RunnablePassthrough.assign(specs_json=prompt_transform | llm | StrOutputParser())
    
    # Step 3: Logistics. Input includes {specs_json} -> Output adds {logistic_json}
    | RunnablePassthrough.assign(logistic_json=prompt_logistic | llm | StrOutputParser())
    
    # Step 4: Order Details. Input includes {logistic_json} AND original {order_*} variables
    | prompt_order
    | llm
    | StrOutputParser()
)

# --- Run the Chain ---
input_data = {
    "text_input": "The new laptop model features a 3.5 GHz octa-core processor, 16GB of RAM, and a 1TB NVMe SSD. The laptop also has a 14-inch display with a resolution of 1920 x 1080 pixels. The laptop will be sended to the customer in 5 days by DHL Delivery.",
    "logistic_json":{
        "order_id": "12345",
        "order_date": "2025-12-15",
        "order_status": "Pending"
    }
}

# Execute the chain with the input dictionary containing all variables.
final_result = full_chain.invoke(input_data)

print("\n--- Final JSON Output ---")
print(final_result)


--- Final JSON Output ---
{
    "specifications": {
        "cpu": "3.5 GHz octa-core",
        "memory": "16GB",
        "storage": "1TB NVMe SSD"
    },
    "shipping_options": [
        {
            "provider": "DHL",
            "shipping_cost": "$10",
            "delivery_days": 3
        },
        {
            "provider": "FedEx",
            "shipping_cost": "$12",
            "delivery_days": 2
        },
        {
            "provider": "UPS",
            "shipping_cost": "$15",
            "delivery_days": 4
        }
    ],
    "selected_provider": {
        "provider": "DHL",
        "send_date": "2022-10-25",
        "estimated_date_delivery_received": "2022-10-28"
    },
    "order_details": {
        "Order_id": "12345",
        "Order_date": "2022-10-24",
        "Order_status": "Processing"
    }
}
